In [1]:
### Check for connected runtime (GPU or not)
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Wed Feb 18 08:38:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
### Check for more RAM
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 179.4 gigabytes of available RAM

You are using a high-RAM runtime!


In [3]:
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))

NameError: name 'torch' is not defined

##### Mount Drive into Collab

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##### Path to Model Repository

In [5]:
%cd /content/drive/MyDrive/Colab_Notebooks/AnySat

/content/drive/MyDrive/Colab_Notebooks/AnySat


##### Installing Model requirements

In [6]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 84.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.2/308.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 7.6 MB/s eta 0:00:00


# [ORIGINAL PART]

# AnySat Guide

#### Simple Usage

In [7]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import rasterio
from datetime import datetime
import h5py
import time
import psutil
import matplotlib.pyplot as plt
from IPython.display import clear_output

## AnySat is available through
### 1) PyTorch Hub
#### or
### 2) local repository

In [8]:
# 1) PyTorch Hub
#model = torch.hub.load('gastruc/anysat', 'anysat', pretrained=True, force_reload=True, flash_attn=False, trust_repo='check')

# 2) local repo
from hubconf import AnySat

model = AnySat.from_pretrained('base', flash_attn=False) #Set flash_attn=True if you have flash-attn module installed (url flash attn)
#device = "cuda" If you want to run on GPU default is cpu

Downloading: "https://huggingface.co/g-astruc/AnySat/resolve/main/models/AnySat.pth" to /root/.cache/torch/hub/checkpoints/AnySat.pth


100%|██████████| 480M/480M [00:09<00:00, 53.6MB/s]


### Added code to assure that every part of the Model runs on GPU

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the entire model and its submodules to the GPU
model = model.to(device)

# Iterate through all model parameters and move them to the GPU if not already there
for param in model.parameters():
    if param.device != device:
        param.data = param.data.to(device)

# Check for any remaining submodules not on the GPU
for name, module in model.named_modules():
    if next(module.parameters(), None) is not None and next(module.parameters()).device != device:  # Checks if module has parameters
        module.to(device)
        print(f"Module '{name}' moved to {device}")

Module '' moved to cuda
Module 'spatial_encoder' moved to cuda
Module 'spatial_encoder.predictor_blocks' moved to cuda
Module 'spatial_encoder.predictor_blocks.0' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.norm1' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.qkv' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.proj' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.rpe_k' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.norm2' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp.fc1' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp.fc2' moved to cuda
Module 'spatial_encoder.predictor_blocks.1' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.norm1' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.attn' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.attn.q

# Feature Extraction

### Complete Folder

In [12]:
import os
import torch
import glob
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

# 🔧 Configuration

feature_set = 'test' # either 'train' or 'test'
NUM_FILES = 5      # either 8688(+1) for 'train' or 2772(+1) for 'test'
ORBIT = 'asc'        # either 'asc' or 'desc'
OUTPUT_TYPE = 'dense'  # Either 'tile', 'patch', 'dense'

BATCH_SIZE = 4        # ✅ Set your optimal batch size here
NUM_WORKERS = 8       # ✅ Set your optimal worker count here
PATCH_SIZE = 80       # ✅ Set your optimal patch size here (320 --> Out-of-Memory Error with A100 Extended RAM (89GB))

# 📁 Paths

INPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/3_channels_VV_VH_ratio/NORMALIZED_{feature_set.upper()}_FEATURES_s1_{ORBIT.lower()}_July"
OUTPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/3_channels_VV_VH_ratio/ENCODED_{feature_set.upper()}_FEATURES_s1_{ORBIT}_July_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
existing_outputs = set(os.listdir(OUTPUT_FOLDER))

model_used = model  # your pretrained model should already be defined

# 📦 Dataset
class TensorDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        tensor = torch.load(path).float()  # Shape: [2, 256, 256]
        filename = os.path.basename(path)
        return tensor, filename

# 📁 Load file paths
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))[:NUM_FILES]
print(len(all_paths))
#all_paths = [sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))[-1]]

# Filter input tensor paths to exclude already-processed files
filtered_paths = []
for path in all_paths:
    original_name = os.path.basename(path)
    cleaned_name = original_name.removeprefix("normalized_")
    expected_output_name = f"Features_{ORBIT}_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}_{cleaned_name}"
    if expected_output_name not in existing_outputs:
        filtered_paths.append(path)

print(f"✅ {len(filtered_paths)} of {len(all_paths)} files will be processed (skipping existing ones)")

#dataset = TensorDataset(all_paths)
dataset = TensorDataset(filtered_paths)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# 📤 Inference & Save
with torch.no_grad():
    for batch, filenames in tqdm(loader, desc="Extracting features"):
        batch = batch.unsqueeze(1).to(device)  # [B, 1, 2, 256, 256]
        dates = torch.tensor([196] * batch.shape[0]).to(device)

        data = {
            "s1": batch,
            "s1_dates": dates
        }

        features = model_used(data, patch_size=PATCH_SIZE, output=OUTPUT_TYPE)  # Feature tensor output from your model

        # 💾 Save infered encoded features
        for i in range(len(filenames)):
            original_name = filenames[i]
            cleaned_name = original_name.removeprefix("normalized_")
            output_path = os.path.join(OUTPUT_FOLDER, f"Features_{ORBIT}_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}_{cleaned_name}")

            torch.save(features[i].cpu(), output_path)
            #print(f"✅ Saved {output_path}")



        # 🧹 Memory cleanup
        del batch, filenames, data, features
        torch.cuda.empty_cache()

print("✅ All encoded features extracted and saved.")

5
✅ 5 of 5 files will be processed (skipping existing ones)


Extracting features:   0%|          | 0/2 [00:00<?, ?it/s]

✅ All encoded features extracted and saved.
